# Objetivo 1 · Visor de predicciones de nnU-Net
**Reto CIMAT / BraTS 2018**

Muestra, corte por corte, la resonancia junto a la segmentación real y la predicción del modelo.

- **Parte A: pacientes de validación del fold 0.** El modelo **nunca los vio** al entrenar y tenemos su segmentación real para comparar. No hace falta GPU: nnU-Net ya guardó estas predicciones al final del entrenamiento.
- **Parte B: un caso externo**, que debe venir en formato BraTS (ver los requisitos ahí). Esta parte sí necesita GPU.

El modelo de prueba entrenó solo **5 épocas**, así que se esperan errores visibles, sobre todo en el realce (ET). Para ver otro modelo, cambia `MODELO` en la celda siguiente.

Requisitos: haber corrido `prueba_colab.ipynb` (los resultados quedaron en Drive) y tener `Dataset501_BraTS2018.zip` en `MyDrive/reto_cimat/`.

## 1. Configuración

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/reto_cimat'
MODELO = 'nnUNetTrainer_5epochs'      # cambiar al entrenar el modelo real (p. ej. nnUNetTrainer_100epochs)
FOLD = 0

os.environ['nnUNet_raw'] = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results'] = f'{DRIVE}/nnUNet_results'

RES = f"{os.environ['nnUNet_results']}/Dataset501_BraTS2018/{MODELO}__nnUNetPlans__3d_fullres/fold_{FOLD}"
VAL = f'{RES}/validation'
assert os.path.exists(f'{VAL}/summary.json'), f'No encuentro las predicciones de validacion en {VAL}'
print('modelo:', RES)

## 2. Extraer del zip solo los pacientes de validación del fold

In [ ]:
%%time
import json, zipfile
LOCAL = '/content/visor'
with zipfile.ZipFile(f'{DRIVE}/Dataset501_BraTS2018.zip') as z:
    splits = json.loads(z.read('Dataset501_BraTS2018/splits_final.json'))
    val_ids = splits[FOLD]['val']
    quiero = {f'Dataset501_BraTS2018/labelsTr/{c}.nii.gz' for c in val_ids} | \
             {f'Dataset501_BraTS2018/imagesTr/{c}_{k:04d}.nii.gz' for c in val_ids for k in range(4)}
    for nombre in quiero:
        z.extract(nombre, LOCAL)
DATOS = f'{LOCAL}/Dataset501_BraTS2018'
print(len(val_ids), 'pacientes de validacion extraidos')

## 3. Funciones de dibujo

In [ ]:
import glob, json, os
import pandas as pd
import functools
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Etiquetas nnU-Net (D23): 1 edema, 2 necrosis / no realzado, 3 realce
COLORES = ListedColormap([(0, 0, 0, 0), (0.1, 0.8, 0.2, 1), (0.9, 0.1, 0.1, 1), (1.0, 0.85, 0.0, 1)])
LEYENDA = [Patch(color=COLORES(1), label='edema'),
           Patch(color=COLORES(2), label='necrosis / no realzado'),
           Patch(color=COLORES(3), label='realce (ET)')]
MODALIDADES = {'T1': '0000', 'T1ce': '0001', 'T2': '0002', 'FLAIR': '0003'}


@functools.lru_cache(maxsize=8)
def cargar(ruta):
    return np.asanyarray(nib.load(ruta).dataobj)


def regiones(seg):
    """Volumen en cm3 (voxel de 1 mm3) de WT, TC y ET con etiquetas nnU-Net."""
    return {'WT': (seg >= 1).sum() / 1000, 'TC': (seg >= 2).sum() / 1000, 'ET': (seg == 3).sum() / 1000}


def corte_con_mas_tumor(seg):
    return int(np.argmax((seg > 0).sum(axis=(0, 1)))) if seg.any() else seg.shape[2] // 2


def dibujar(imagen, pred, gt=None, z=None, titulo='', alfa=0.45):
    """Corte axial z: MRI | MRI + real (si hay) | MRI + prediccion."""
    if z is None:
        z = corte_con_mas_tumor(gt if gt is not None else pred)
    rot = lambda a: np.rot90(a[:, :, z])
    paneles = [('MRI', None)]
    if gt is not None:
        paneles.append(('Segmentación real', gt))
    paneles.append(('Predicción del modelo', pred))

    fig, ejes = plt.subplots(1, len(paneles), figsize=(5 * len(paneles), 5.4))
    for eje, (nombre, seg) in zip(np.atleast_1d(ejes), paneles):
        eje.imshow(rot(imagen), cmap='gray')
        if seg is not None:
            eje.imshow(rot(seg), cmap=COLORES, vmin=0, vmax=3, alpha=alfa, interpolation='nearest')
        eje.set_title(nombre)
        eje.axis('off')
    fig.legend(handles=LEYENDA, loc='lower center', ncol=3, frameon=False)
    fig.suptitle(f'{titulo}   corte axial {z}')
    fig.tight_layout(rect=(0, 0.06, 1, 0.95))
    return fig



def dice_region(a, b):
    """Dice; NaN si la region no existe ni en la real ni en la prediccion."""
    s = a.sum() + b.sum()
    return np.nan if s == 0 else 2 * np.logical_and(a, b).sum() / s

def cargar_dice():
    """Dice por paciente y region (WT/TC/ET) de los casos con prediccion en VAL."""
    casos = sorted(os.path.basename(f)[:-7] for f in glob.glob(f'{VAL}/*.nii.gz'))
    try:
        resumen = json.load(open(f'{VAL}/summary.json'))
        claves = {'WT': '(1, 2, 3)', 'TC': '(2, 3)', 'ET': '(3,)'}
        filas = [{'paciente': os.path.basename(c['reference_file']).replace('.nii.gz', ''),
                  **{r: c['metrics'][k]['Dice'] for r, k in claves.items()}}
                 for c in resumen['metric_per_case']]
        tabla = pd.DataFrame(filas).set_index('paciente')
        assert set(casos) <= set(tabla.index)
        print('Dice leido de summary.json')
    except Exception as e:
        print(f'summary.json no utilizable ({type(e).__name__}: {e}); calculando Dice con las mascaras...')
        filas = []
        for c in casos:
            gt, pred = cargar(f'{DATOS}/labelsTr/{c}.nii.gz'), cargar(f'{VAL}/{c}.nii.gz')
            filas.append({'paciente': c, 'WT': dice_region(gt >= 1, pred >= 1),
                          'TC': dice_region(gt >= 2, pred >= 2), 'ET': dice_region(gt == 3, pred == 3)})
        tabla = pd.DataFrame(filas).set_index('paciente')
    return tabla.loc[casos].sort_values('WT')


## 4. Dice por paciente
Se toma de `summary.json`, que nnU-Net generó al validar. Si no se puede leer, se calcula directamente con las máscaras (tarda uno o dos minutos). Ordenado de peor a mejor en WT.

In [ ]:
dice = cargar_dice()
print(f'{len(dice)} pacientes | media:', dice.mean().round(3).to_dict(),
      '  (ET = NaN: sin realce ni real ni predicho)')
dice.round(3)

## 5. Parte A · Visor de los pacientes de validación
Elige paciente, modalidad y corte. Al cambiar de paciente, el corte salta al que tiene más tumor.

In [ ]:
import ipywidgets as w
from IPython.display import display

def visor(opciones, ruta_img, ruta_pred, ruta_gt=None):
    """opciones: [(etiqueta, id)]; ruta_*: funciones id -> ruta (ruta_img recibe tambien el canal)."""
    caso_w = w.Dropdown(options=opciones, description='Paciente', layout=w.Layout(width='620px'))
    mod_w = w.ToggleButtons(options=list(MODALIDADES), value='FLAIR', description='Modalidad')
    z_w = w.IntSlider(min=0, max=154, description='Corte', continuous_update=False)
    alfa_w = w.FloatSlider(value=0.45, min=0, max=1, step=0.05, description='Opacidad',
                           continuous_update=False)

    def al_cambiar(cambio):
        ref = ruta_gt(cambio['new']) if ruta_gt else ruta_pred(cambio['new'])
        z_w.value = corte_con_mas_tumor(cargar(ref))
    caso_w.observe(al_cambiar, names='value')
    al_cambiar({'new': caso_w.value})

    def mostrar(caso, mod, z, alfa):
        img = cargar(ruta_img(caso, MODALIDADES[mod]))
        pred = cargar(ruta_pred(caso))
        gt = cargar(ruta_gt(caso)) if ruta_gt else None
        dibujar(img, pred, gt, z, titulo=caso, alfa=alfa)
        plt.show()
        vols = {'predicción': regiones(pred)}
        if gt is not None:
            vols = {'real': regiones(gt), **vols}
        print('volumen (cm³):')
        print(pd.DataFrame(vols).T.round(1).to_string())

    salida = w.interactive_output(mostrar, {'caso': caso_w, 'mod': mod_w, 'z': z_w, 'alfa': alfa_w})
    display(w.VBox([caso_w, mod_w, w.HBox([z_w, alfa_w])]), salida)

if 'dice' not in globals():      # por si la seccion 4 no se ejecuto
    dice = cargar_dice()
fmt = lambda x: '—' if pd.isna(x) else f'{x:.2f}'
opciones = [(f'{p}   WT {fmt(r.WT)} · TC {fmt(r.TC)} · ET {fmt(r.ET)}', p) for p, r in dice.iterrows()]
visor(opciones,
      ruta_img=lambda c, k: f'{DATOS}/imagesTr/{c}_{k}.nii.gz',
      ruta_pred=lambda c: f'{VAL}/{c}.nii.gz',
      ruta_gt=lambda c: f'{DATOS}/labelsTr/{c}.nii.gz')

## 6. ¿El error depende del tamaño del tumor?
Revisa con **todos** los pacientes de validación si los tumores pequeños se predicen peor.
- **Error relativo de volumen** = |predicho − real| / real. Un tumor chico tiene más superficie por unidad de volumen, así que el mismo error de borde pesa más: se espera que el error relativo crezca al bajar el volumen aunque el modelo sea igual de preciso en todos.
- **Dice**: mide la superposición, no solo el volumen.
- **ρ de Spearman** entre volumen real y cada medida: con ρ < 0 en el error y ρ > 0 en el Dice, los tumores pequeños salen peor. Con 57 pacientes, solo un p < 0.05 sugiere algo más que ruido.

Hay que repetirlo con el modelo final: el de 5 épocas no es representativo.

In [ ]:
from scipy.stats import spearmanr

filas = []
for c in dice.index:
    gt, pred = cargar(f'{DATOS}/labelsTr/{c}.nii.gz'), cargar(f'{VAL}/{c}.nii.gz')
    vr, vp = regiones(gt), regiones(pred)
    for r in ('WT', 'TC', 'ET'):
        filas.append({'paciente': c, 'region': r, 'real_cm3': vr[r], 'pred_cm3': vp[r],
                      'Dice': dice.loc[c, r]})
vol = pd.DataFrame(filas)
# Sin volumen real no hay error relativo (p. ej. LGG sin ET)
vol['error_rel'] = np.where(vol.real_cm3 > 0,
                            (vol.pred_cm3 - vol.real_cm3).abs() / vol.real_cm3.where(vol.real_cm3 > 0), np.nan)

res = []
for r in ('WT', 'TC', 'ET'):
    x = vol[(vol.region == r) & (vol.real_cm3 > 0)]
    for medida in ('error_rel', 'Dice'):
        y = x.dropna(subset=[medida])
        rho, pval = spearmanr(y.real_cm3, y[medida])
        res.append({'region': r, 'medida': medida, 'n': len(y), 'rho': round(rho, 2), 'p': round(pval, 4)})
print(pd.DataFrame(res).to_string(index=False))

fig, ejes = plt.subplots(2, 3, figsize=(13, 7), sharex='col')
for j, r in enumerate(('WT', 'TC', 'ET')):
    x = vol[(vol.region == r) & (vol.real_cm3 > 0)]
    for i, (medida, nombre) in enumerate((('error_rel', 'error relativo de volumen'), ('Dice', 'Dice'))):
        eje = ejes[i, j]
        eje.scatter(x.real_cm3, x[medida] * (100 if medida == 'error_rel' else 1),
                    s=36, color='#2a78d6', edgecolor='white', linewidth=1, alpha=0.85)
        eje.set_xscale('log')
        eje.grid(color='#e5e5e2', linewidth=0.8)
        eje.spines[['top', 'right']].set_visible(False)
        eje.set_ylabel(f'{nombre} (%)' if medida == 'error_rel' else nombre, color='#52514e')
        if i == 0:
            eje.set_title(r)
        if i == 1:
            eje.set_xlabel('volumen real (cm³, escala log)', color='#52514e')
fig.suptitle(f'Error de la prediccion segun el tamano del tumor ({MODELO}, fold {FOLD})')
fig.tight_layout()
plt.show()

print('\nLos 8 mayores errores relativos de WT:')
print(vol[vol.region == 'WT'].nlargest(8, 'error_rel')[['paciente', 'real_cm3', 'pred_cm3', 'error_rel', 'Dice']]
      .round(2).to_string(index=False))

## 7. Parte B · Caso externo (opcional, necesita GPU)

**El modelo solo funciona con imágenes en formato BraTS.** Una resonancia clínica cualquiera no sirve sin preprocesarla antes:
- las **4 modalidades** del mismo paciente: T1, T1ce (con contraste), T2 y FLAIR;
- **sin cráneo** (skull-stripped), **co-registradas** entre sí y al atlas SRI24;
- voxel de **1 mm** y tamaño **240 × 240 × 155**.

**Dónde conseguir casos externos ya preprocesados:**
- **BraTS 2018, conjunto de validación** (66 pacientes, sin segmentación pública): mismo formato y ninguno está en nuestro entrenamiento. Se pide con registro en el portal de BraTS (CBICA).
- **BraTS-Africa (2023, en TCIA):** formato BraTS, población distinta, ideal para probar la generalización. Ojo: su convención de etiquetas usa 3 = realce.
- **Evitar BraTS 2019, 2020 y 2021:** incluyen los casos de 2018, así que el modelo podría haberlos visto.
- Para una resonancia clínica cruda, el preprocesamiento se hace con BraTS Toolkit o CaPTk.

**Cómo usarla:** sube a `MyDrive/reto_cimat/caso_nuevo/` cuatro archivos llamados exactamente `t1.nii.gz`, `t1ce.nii.gz`, `t2.nii.gz` y `flair.nii.gz`.

In [ ]:
import numpy as np, nibabel as nib, shutil
CASO = f'{DRIVE}/caso_nuevo'
ENTRADA, SALIDA = '/content/caso_in', '/content/caso_out'
os.makedirs(ENTRADA, exist_ok=True); os.makedirs(SALIDA, exist_ok=True)

imgs = {m: nib.load(f'{CASO}/{m}.nii.gz') for m in ('t1', 't1ce', 't2', 'flair')}
ref = imgs['t1']
avisos = []
for m, im in imgs.items():
    if im.shape != ref.shape or not np.allclose(im.affine, ref.affine, atol=1e-3):
        avisos.append(f'{m}: forma o geometria distinta de t1 (no estan co-registradas)')
if ref.shape != (240, 240, 155):
    avisos.append(f'tamano {ref.shape}, BraTS usa (240, 240, 155)')
if not np.allclose(ref.header.get_zooms()[:3], 1, atol=1e-2):
    avisos.append(f'voxel {ref.header.get_zooms()[:3]} mm, BraTS usa 1 mm')
fondo = float((np.asanyarray(ref.dataobj) == 0).mean())
if fondo < 0.5:
    avisos.append(f'solo {fondo:.0%} de fondo en 0: probablemente conserva el craneo')
print('\n'.join('AVISO: ' + a for a in avisos) if avisos else 'formato BraTS: ok')

for k, m in enumerate(('t1', 't1ce', 't2', 'flair')):
    shutil.copy(f'{CASO}/{m}.nii.gz', f'{ENTRADA}/caso_{k:04d}.nii.gz')

In [ ]:
%%time
!pip install -q nnunetv2
!nnUNetv2_predict -i {ENTRADA} -o {SALIDA} -d 501 -c 3d_fullres -tr {MODELO} -f {FOLD}

In [ ]:
visor([('caso nuevo', 'caso')],
      ruta_img=lambda c, k: f'{ENTRADA}/{c}_{k}.nii.gz',
      ruta_pred=lambda c: f'{SALIDA}/{c}.nii.gz')